In [1]:
#essentials
import pandas as pd
import numpy as np
import scipy as sns
import matplotlib.pyplot as plt
%matplotlib inline

### Some Updates to make: [Will perform these changes after seeing the current model result]

Before model training, i'd do after gaining insigts from visaulisation:

- Clip or winsorize the worst outliers in comment_len, engagement_score, num_ques, and num_!.

- Use log1p on heavy-tailed count-like features.

- Convert created_hour to cyclic features (sin, cos) instead of treating it as a linear integer.

- Keep missingness as an explicit category/indicator for categorical fields like gender, rather than hiding it.

- Drop or de-prioritize flat features like post_comments_count unless they prove useful in cross-validation.

- Handle class imbalance with stratified splits and class weights; evaluate with macro-F1, not accuracy.

One more thing: votes_ratio may work better binned or paired with vote volume than as a raw float. As it stands, it behaves more like a quasi-discrete signal than a smooth numeric feature.

In [2]:
df = pd.read_csv("train_df.csv")

KeyboardInterrupt: 

In [ ]:
y_train = df['label'].values
y_train

array([2., 0., 2., ..., 3., 1., 0.], shape=(198000,))

In [ ]:
from scipy import sparse

data = np.load('X_train_.npz', allow_pickle=True)

In [ ]:
from scipy.sparse import load_npz
X_train_df = load_npz('X_train_.npz')

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train_df, 
    y_train, 
    test_size=0.20, 
    random_state=42, 
    stratify=y_train
)

In [ ]:
print(f"Training data shape: {X_train_split.shape}")
print(f"Validation data shape: {X_val_split.shape}")
# print(f"Validation data shape: {X_train_split.shape[1]+ X_val_split.shape[1]}")

Training data shape: (158400, 20045)
Validation data shape: (39600, 20045)


In [ ]:
unique, counts = np.unique(y_val_split, return_counts=True)
print("\nClass distribution in Validation set:")
for label, count in zip(unique, counts):
    print(f"Label {label}: {count} rows")


Class distribution in Validation set:
Label 0.0: 22835 rows
Label 1.0: 3183 rows
Label 2.0: 12488 rows
Label 3.0: 1094 rows


### **Model Training**

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

lr = LogisticRegression(max_iter= 1000, class_weight='balanced', n_jobs=-1)


In [ ]:
lr.fit(X_train_split, y_train_split)
y_pred = lr.predict(X_val_split)

print(classification_report(y_val_split, y_pred))
print("f1_sctrore: ", f1_score(y_val_split, y_pred, average='macro'))

In [ ]:
#imports for model
#ref- collab code!

In [ ]:
from scipy.sparse import load_npz

X_train_split, X_val_split, y_train_split, y_val_split = load_npz('X_train_final.npz'), load_npz('X_val_final.npz'), np.load('y_train.npy'), np.load('y_val.npy')

### Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

lr = LogisticRegression(max_iter= 1000, class_weight='balanced', n_jobs=-1)


In [ ]:
lr.fit(X_train_split, y_train_split)
y_pred = lr.predict(X_val_split)
print(classification_report(y_val_split, y_pred))
print("f1_sctrore: ", f1_score(y_val_split, y_pred, average='macro'))

In [ ]:
X_train_final, X_val_final = load_npz('X_train_final.npz'), load_npz('X_val_final.npz'),

y_train = np.load('y_train.npy')
y_val = np.load('y_val.npy')

In [ ]:
#to verfiy is theres' any leakage or not.. 
import numpy as np
y_shuffled = np.random.permutation(y_train)
lr.fit(X_train_final, y_shuffled)
y_pred = lr.predict(X_val_final)

from sklearn.metrics import f1_score
print("Macro F1 (shuffled):", f1_score(y_val, y_pred, average='macro'))

from collab:

              precision    recall  f1-score   support

           0       0.98      0.93      0.96     22835
           1       0.67      0.84      0.75      3183
           2       0.88      0.85      0.87     12488
           3       0.46      0.77      0.58      1094

    accuracy                           0.90     39600
   macro avg       0.75      0.85      0.79     39600
weighted avg       0.91      0.90      0.90     39600

f1_sctrore:  0.7865856547693546


### LightGBM

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, f1_score

In [ ]:
lgb = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=-1,
    class_weight='balanced',
    n_jobs=-1
)
lgb.fit(X_train_split, y_train_split)
y_pred = lgb.predict(X_val_split)

print(classification_report(y_val_split, y_pred))
print("Macro F1:", f1_score(y_val_split, y_pred, average='macro'))

From collab: 

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 56.329616 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 874457
[LightGBM] [Info] Number of data points in the train set: 158400, number of used features: 19959
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/lightgbm/basic.py:1238: UserWarning: Converting data to scipy sparse matrix.
  _log_warning("Converting data to scipy sparse matrix.")
              precision    recall  f1-score   support

           0       0.98      0.94      0.96     22835
           1       0.68      0.86      0.76      3183
           2       0.90      0.82      0.86     12488
           3       0.40      0.81      0.54      1094

    accuracy                           0.90     39600
   macro avg       0.74      0.86      0.78     39600
weighted avg       0.92      0.90      0.90     39600

Macro F1: 0.7807100332284211
